In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import gc

# 1. Tải dữ liệu
df_booking = pd.read_csv('../data/processed/Cleaned_Airline_Review_Full.csv')
target_col = 'booking_complete' 

if target_col in df_booking.columns:
    y = df_booking[target_col]
    X = df_booking.drop(columns=[target_col])
    
    # Ép kiểu dữ liệu chữ sang số
    cat_cols = X.select_dtypes(include=['object']).columns
    for col in cat_cols:
        X[col] = X[col].astype('category').cat.codes
        
    del df_booking
    gc.collect() 
    
    # 2. CHIA TẬP DỮ LIỆU
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)
    
    # 3. CHUẨN HÓA DỮ LIỆU (Feature Scaling - Giúp sửa lỗi ConvergenceWarning)
    scaler = StandardScaler()
    # Chỉ fit (tìm quy luật) trên tập Train, sau đó áp dụng (transform) quy luật đó lên Train và Val
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # 4. HUẤN LUYỆN VÀ ĐÁNH GIÁ SÂU
    print("\n" + "="*40)
    print("MÔ HÌNH: LOGISTIC REGRESSION")
    print("="*40)
    log_model = LogisticRegression(max_iter=1000)
    log_model.fit(X_train_scaled, y_train) # Dùng dữ liệu đã scale
    y_val_pred_log = log_model.predict(X_val_scaled)
    
    print("1. Accuracy:", round(accuracy_score(y_val, y_val_pred_log), 4))
    print("\n2. Classification Report (Đánh giá sâu):")
    print(classification_report(y_val, y_val_pred_log))
    
    
    print("\n" + "="*40)
    print("MÔ HÌNH: DECISION TREE")
    print("="*40)
    # Cây quyết định không bị ảnh hưởng bởi thang đo, nên dùng dữ liệu gốc cũng được
    # Nhưng ta dùng max_depth=7 để xem nó có học được nhiều hơn không
    tree_model = DecisionTreeClassifier(random_state=42, max_depth=7)
    tree_model.fit(X_train, y_train)
    y_val_pred_tree = tree_model.predict(X_val)
    
    print("1. Accuracy:", round(accuracy_score(y_val, y_val_pred_tree), 4))
    print("\n2. Classification Report (Đánh giá sâu):")
    print(classification_report(y_val, y_val_pred_tree))

else:
    print(f"Lỗi: Không tìm thấy cột '{target_col}'.")


MÔ HÌNH: LOGISTIC REGRESSION
1. Accuracy: 0.8444

2. Classification Report (Đánh giá sâu):
              precision    recall  f1-score   support

           0       0.84      1.00      0.92      6659
           1       0.00      0.00      0.00      1227

    accuracy                           0.84      7886
   macro avg       0.42      0.50      0.46      7886
weighted avg       0.71      0.84      0.77      7886


MÔ HÌNH: DECISION TREE
1. Accuracy: 0.8421

2. Classification Report (Đánh giá sâu):
              precision    recall  f1-score   support

           0       0.85      0.99      0.91      6659
           1       0.43      0.05      0.08      1227

    accuracy                           0.84      7886
   macro avg       0.64      0.52      0.50      7886
weighted avg       0.78      0.84      0.78      7886



c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

SMOTE (Synthetic Minority Over-sampling Technique)

In [14]:
# Cần chạy lệnh 'pip install imbalanced-learn' trong terminal trước khi chạy đoạn code này
from imblearn.over_sampling import SMOTE
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import gc

# 1. Tải và xử lý lại dữ liệu nhanh
df_booking = pd.read_csv('../data/processed/Cleaned_Passenger_Booking.csv')
target_col = 'booking_complete' 
y = df_booking[target_col]
X = df_booking.drop(columns=[target_col])

cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    X[col] = X[col].astype('category').cat.codes
    
del df_booking
gc.collect()

# 2. CHIA TẬP DỮ LIỆU
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

# 3. CHUẨN HÓA DỮ LIỆU
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# 4. ÁP DỤNG SMOTE ĐỂ CÂN BẰNG DỮ LIỆU TRÊN TẬP TRAIN
print("\n--- Đang cân bằng dữ liệu bằng SMOTE ---")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print(f"Số lượng mẫu ban đầu: Lớp 0: {sum(y_train==0)}, Lớp 1: {sum(y_train==1)}")
print(f"Số lượng mẫu sau SMOTE: Lớp 0: {sum(y_train_smote==0)}, Lớp 1: {sum(y_train_smote==1)}")

# 5. HUẤN LUYỆN LẠI SAU KHI CÂN BẰNG
print("\n" + "="*40)
print("MÔ HÌNH: LOGISTIC REGRESSION (SAU SMOTE)")
print("="*40)
log_model_smote = LogisticRegression(max_iter=1000)
log_model_smote.fit(X_train_smote, y_train_smote)
y_val_pred_log_smote = log_model_smote.predict(X_val_scaled)

print("1. Accuracy:", round(accuracy_score(y_val, y_val_pred_log_smote), 4))
print("\n2. Classification Report:")
print(classification_report(y_val, y_val_pred_log_smote))

print("\n" + "="*40)
print("MÔ HÌNH: DECISION TREE (SAU SMOTE)")
print("="*40)
tree_model_smote = DecisionTreeClassifier(random_state=42, max_depth=7)
tree_model_smote.fit(X_train_smote, y_train_smote) # Dùng dữ liệu SMOTE nhưng không scale cho Decision Tree thì hợp lý hơn, tuy nhiên dùng scaled cũng được. Để đơn giản ta dùng luôn bản scaled.
y_val_pred_tree_smote = tree_model_smote.predict(X_val_scaled)

print("1. Accuracy:", round(accuracy_score(y_val, y_val_pred_tree_smote), 4))
print("\n2. Classification Report:")
print(classification_report(y_val, y_val_pred_tree_smote))


--- Đang cân bằng dữ liệu bằng SMOTE ---
Số lượng mẫu ban đầu: Lớp 0: 26799, Lớp 1: 4741
Số lượng mẫu sau SMOTE: Lớp 0: 26799, Lớp 1: 26799

MÔ HÌNH: LOGISTIC REGRESSION (SAU SMOTE)
1. Accuracy: 0.6121

2. Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.61      0.73      6659
           1       0.22      0.61      0.33      1227

    accuracy                           0.61      7886
   macro avg       0.56      0.61      0.53      7886
weighted avg       0.79      0.61      0.67      7886


MÔ HÌNH: DECISION TREE (SAU SMOTE)
1. Accuracy: 0.6846

2. Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.69      0.79      6659
           1       0.28      0.67      0.40      1227

    accuracy                           0.68      7886
   macro avg       0.60      0.68      0.59      7886
weighted avg       0.82      0.68      0.73      7886



Cải thiện các chỉ số bằng cách sử dụng GridSearchCV

In [15]:
from sklearn.model_selection import GridSearchCV
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo đỏ cho đỡ rối mắt

print("🚀 ĐANG KHỞI ĐỘNG GRID SEARCH CV (Vui lòng đợi 1-3 phút)...\n")

# ==========================================
# 1. TỐI ƯU HÓA DECISION TREE
# ==========================================
print("1. Đang dò tìm tham số tốt nhất cho Cây Quyết Định...")
tree_params = {
    'max_depth': [5, 10, 15, 20],        # Thử các độ sâu khác nhau
    'min_samples_split': [10, 20, 50],   # Số mẫu tối thiểu để chia nhánh
    'criterion': ['gini', 'entropy']     # Thuật toán đo lường độ tinh khiết
}

# n_jobs=-1 giúp huy động 100% sức mạnh CPU của laptop để chạy nhanh hơn
grid_tree = GridSearchCV(DecisionTreeClassifier(random_state=42), 
                         param_grid=tree_params, 
                         cv=5, 
                         scoring='f1', # Ép mô hình phải tập trung vào việc bắt trúng khách hàng (Lớp 1)
                         n_jobs=-1)

grid_tree.fit(X_train_smote, y_train_smote)
best_tree = grid_tree.best_estimator_

y_val_pred_best_tree = best_tree.predict(X_val_scaled)
print(f"-> Đã tìm thấy Cây tối ưu: {grid_tree.best_params_}")
print("\nBÁO CÁO KẾT QUẢ DECISION TREE (ĐÃ TỐI ƯU):")
print("Accuracy:", round(accuracy_score(y_val, y_val_pred_best_tree), 4))
print(classification_report(y_val, y_val_pred_best_tree))

# ==========================================
# 2. TỐI ƯU HÓA LOGISTIC REGRESSION
# ==========================================
print("\n" + "="*40)
print("2. Đang dò tìm tham số tốt nhất cho Hồi Quy Logistic...")
log_params = {
    'C': [0.01, 0.1, 1, 10], # Thử nghiệm các mức độ kiểm soát nhiễu (Regularization)
    'solver': ['liblinear', 'lbfgs']
}

grid_log = GridSearchCV(LogisticRegression(max_iter=2000, random_state=42), 
                        param_grid=log_params, 
                        cv=5, 
                        scoring='f1', 
                        n_jobs=-1)

grid_log.fit(X_train_smote, y_train_smote)
best_log = grid_log.best_estimator_

y_val_pred_best_log = best_log.predict(X_val_scaled)
print(f"-> Đã tìm thấy Logistic tối ưu: {grid_log.best_params_}")
print("\nBÁO CÁO KẾT QUẢ LOGISTIC REGRESSION (ĐÃ TỐI ƯU):")
print("Accuracy:", round(accuracy_score(y_val, y_val_pred_best_log), 4))
print(classification_report(y_val, y_val_pred_best_log))

🚀 ĐANG KHỞI ĐỘNG GRID SEARCH CV (Vui lòng đợi 1-3 phút)...

1. Đang dò tìm tham số tốt nhất cho Cây Quyết Định...
-> Đã tìm thấy Cây tối ưu: {'criterion': 'gini', 'max_depth': 20, 'min_samples_split': 10}

BÁO CÁO KẾT QUẢ DECISION TREE (ĐÃ TỐI ƯU):
Accuracy: 0.7777
              precision    recall  f1-score   support

           0       0.88      0.85      0.87      6659
           1       0.32      0.37      0.34      1227

    accuracy                           0.78      7886
   macro avg       0.60      0.61      0.60      7886
weighted avg       0.79      0.78      0.78      7886


2. Đang dò tìm tham số tốt nhất cho Hồi Quy Logistic...
-> Đã tìm thấy Logistic tối ưu: {'C': 0.01, 'solver': 'liblinear'}

BÁO CÁO KẾT QUẢ LOGISTIC REGRESSION (ĐÃ TỐI ƯU):
Accuracy: 0.6116
              precision    recall  f1-score   support

           0       0.90      0.61      0.73      6659
           1       0.22      0.61      0.33      1227

    accuracy                           0.61      788

LINEAR REGRESSION

In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# 1. Tải dữ liệu Đánh giá chuyến bay (Review)
df_review = pd.read_csv('../data/processed/Cleaned_Airline_Review.csv')

# 2. Xác định biến mục tiêu (Target) là điểm số Rating
target_col = 'Rating'

if target_col in df_review.columns:
    y = df_review[target_col]
    X = df_review.drop(columns=[target_col])
    
    # 3. Ép kiểu dữ liệu (Chuyển Text sang Số)
    cat_cols = X.select_dtypes(include=['object']).columns
    for col in cat_cols:
        X[col] = X[col].astype('category').cat.codes
        
    # 4. Chia tập dữ liệu (Lần này ta chia Train/Test 80-20 là đủ cho Hồi quy tuyến tính)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 5. Chuẩn hóa dữ liệu 
    # Mặc dù Linear Regression không bắt buộc phải scale, nhưng làm việc này 
    # sẽ giúp chúng ta dễ dàng so sánh mức độ quan trọng của các biến sau này
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 6. HUẤN LUYỆN MÔ HÌNH HỒI QUY TUYẾN TÍNH
    print("\n" + "="*45)
    print("MÔ HÌNH: LINEAR REGRESSION (HỒI QUY TUYẾN TÍNH)")
    print("="*45)
    lin_model = LinearRegression()
    lin_model.fit(X_train_scaled, y_train)
    
    # 7. DỰ ĐOÁN VÀ ĐÁNH GIÁ
    y_pred = lin_model.predict(X_test_scaled)
    
    print("Các chỉ số đánh giá hiệu suất mô hình:")
    # MAE: Sai số trung bình tuyệt đối (ví dụ: mô hình đoán lệch thực tế bao nhiêu điểm)
    print(f"- Mean Absolute Error (MAE): {round(mean_absolute_error(y_test, y_pred), 4)}")
    
    # MSE: Sai số bình phương trung bình (phạt nặng các lỗi sai lớn)
    print(f"- Mean Squared Error (MSE): {round(mean_squared_error(y_test, y_pred), 4)}")
    
    # R-squared (R2): Tỷ lệ % độ biến thiên của Rating được giải thích bởi các biến khác
    print(f"- R-squared (R2 Score): {round(r2_score(y_test, y_pred), 4)}")
    
    # 8. Xem trọng số của các tính năng (Tính năng nào ảnh hưởng tới Rating nhiều nhất?)
    print("\nTrọng số các biến (Coefficients):")
    coef_df = pd.DataFrame({'Đặc trưng': X.columns, 'Trọng số': lin_model.coef_})
    # Sắp xếp để xem cái nào tác động mạnh nhất
    print(coef_df.sort_values(by='Trọng số', key=abs, ascending=False).head(5))

else:
    print(f"Lỗi: Không tìm thấy cột '{target_col}'.")


MÔ HÌNH: LINEAR REGRESSION (HỒI QUY TUYẾN TÍNH)
Các chỉ số đánh giá hiệu suất mô hình:
- Mean Absolute Error (MAE): 2.6492
- Mean Squared Error (MSE): 9.3731
- R-squared (R2 Score): 0.0984

Trọng số các biến (Coefficients):
        Đặc trưng  Trọng số
7           Class  0.847694
6  Traveller_type -0.566730
3        Verified  0.245581
5  Review_content -0.241473
1    Flying_month  0.153222


Cải tiến LINEAR REGRESSION

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# 1. Tải dữ liệu Đánh giá
df_review = pd.read_csv('../data/processed/Cleaned_Airline_Review.csv')
target_col = 'Rating'

if target_col in df_review.columns:
    # --- TRICK 1: LOẠI BỎ NHIỄU TEXT ---
    if 'Review_content' in df_review.columns:
        df_review = df_review.drop(columns=['Review_content'])
        print("Đã loại bỏ nhiễu từ cột Review_content.")
        
    y = df_review[target_col]
    X_raw = df_review.drop(columns=[target_col])
    
    # --- TRICK 2: DÙNG ONE-HOT ENCODING ---
    # Chuyển đổi các biến phân loại thành các cột 0/1 độc lập, drop_first=True để tránh bẫy đa cộng tuyến
    X = pd.get_dummies(X_raw, drop_first=True)
    
    # 3. Chia tập Train/Test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # 4. Chuẩn hóa dữ liệu
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 5. HUẤN LUYỆN LẠI LINEAR REGRESSION
    print("\n" + "="*45)
    print("MÔ HÌNH: LINEAR REGRESSION (ĐÃ CẢI TIẾN)")
    print("="*45)
    lin_model_improved = LinearRegression()
    lin_model_improved.fit(X_train_scaled, y_train)
    
    y_pred_improved = lin_model_improved.predict(X_test_scaled)
    
    print("Các chỉ số đánh giá hiệu suất mô hình mới:")
    print(f"- R-squared (R2 Score): {round(r2_score(y_test, y_pred_improved), 4)}")
    print(f"- Mean Absolute Error (MAE): {round(mean_absolute_error(y_test, y_pred_improved), 4)}")
    print(f"- Mean Squared Error (MSE): {round(mean_squared_error(y_test, y_pred_improved), 4)}")

else:
    print(f"Lỗi: Không tìm thấy cột '{target_col}'.")

Đã loại bỏ nhiễu từ cột Review_content.

MÔ HÌNH: LINEAR REGRESSION (ĐÃ CẢI TIẾN)
Các chỉ số đánh giá hiệu suất mô hình mới:
- R-squared (R2 Score): 0.279
- Mean Absolute Error (MAE): 2.2963
- Mean Squared Error (MSE): 7.4956
